# 03 — WAHIS Analytical Table

## Machine Learning-Based Early Warning System for Poultry Disease Outbreaks in Nigeria

This notebook transforms the cleaned WAHIS surveillance records into an analytical table for temporal and machine-learning analysis.

The analytical-table stage will:

1. Load the cleaned WAHIS dataset.
2. Inspect the cleaned structure.
3. Restrict the dataset to the relevant geographic scope.
4. Examine explicitly reported outbreak information.
5. Identify records with usable quantitative surveillance information.
6. Preserve missing outbreak information as missing.
7. Create an explicit observation-status field.
8. Summarise disease and temporal coverage.
9. Examine the geographic structure of the observations.
10. Save the resulting analytical table.

### Critical methodological principle

A record without an explicitly reported outbreak count must not automatically be treated as a negative outbreak observation.

Therefore:

- an explicitly observed outbreak count is treated as observed information;
- an explicitly recorded zero remains zero;
- missing/unavailable outbreak information remains missing.

The analytical table therefore distinguishes **observed surveillance information** from **unobserved or unavailable information**.

This distinction is essential before constructing a supervised machine-learning target.

In [7]:
# ============================================================
# NOTEBOOK 03 — BUILD WAHIS ANALYTICAL TABLE
# ============================================================


from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. DEFINE PROJECT PATHS
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


CLEANED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "wahis_quantitative_cleaned.csv"
)

ANALYTICAL_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

ANALYTICAL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ANALYTICAL_DATA_PATH = (
    ANALYTICAL_DATA_DIR
    / "wahis_analytical_table.csv"
)


# ------------------------------------------------------------
# 2. LOAD CLEANED DATA
# ------------------------------------------------------------
# We deliberately start from the cleaned dataset produced
# by Notebook 02 rather than returning to the raw WAHIS file.

df = pd.read_csv(
    CLEANED_DATA_PATH
)

print("Cleaned WAHIS dataset loaded.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")


# ------------------------------------------------------------
# 3. CHECK COUNTRY COVERAGE
# ------------------------------------------------------------
# The project focuses specifically on poultry disease
# surveillance in Nigeria.
#
# We inspect country values before filtering so that we can
# confirm the geographic scope of the source data.

print("\nCountry distribution:")

display(
    df["country"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 4. FILTER TO NIGERIA
# ------------------------------------------------------------
# Only records belonging to Nigeria are retained for this
# Nigeria-specific early-warning project.
#
# The comparison is case-insensitive and whitespace is removed
# to avoid accidental exclusion caused by formatting.

df["country"] = (
    df["country"]
    .astype("string")
    .str.strip()
)

nigeria_mask = (
    df["country"]
    .str.lower()
    .eq("nigeria")
)

df_nigeria = (
    df.loc[nigeria_mask]
    .copy()
)

print(
    f"\nNigeria records: "
    f"{len(df_nigeria):,}"
)


# ------------------------------------------------------------
# 5. CHECK THE OUTBREAK VARIABLE
# ------------------------------------------------------------
# 'new_outbreaks' is one of the key quantitative surveillance
# variables.
#
# We examine its observed and missing values before creating
# any analytical indicators.

print("\nNew outbreaks — missing values:")
print(
    df_nigeria["new_outbreaks"]
    .isna()
    .sum()
)

print("\nNew outbreaks — observed values:")
print(
    df_nigeria["new_outbreaks"]
    .notna()
    .sum()
)

print("\nNew outbreaks — descriptive statistics:")

display(
    df_nigeria["new_outbreaks"]
    .describe()
)


# ------------------------------------------------------------
# 6. CREATE OBSERVATION STATUS
# ------------------------------------------------------------
# This is an important methodological step.
#
# We explicitly identify whether the outbreak-count field is
# observed or unavailable.
#
# We DO NOT convert missing values into zero.

df_nigeria["outbreak_count_observed"] = (
    df_nigeria["new_outbreaks"]
    .notna()
)


# Create a human-readable status label.

df_nigeria["outbreak_observation_status"] = np.where(
    df_nigeria["outbreak_count_observed"],
    "Observed",
    "Unavailable"
)


# ------------------------------------------------------------
# 7. CHECK EXPLICIT ZERO VALUES
# ------------------------------------------------------------
# Explicit zero outbreak counts remain zero.
#
# They are not treated as missing because zero is an actual
# reported numerical value in the source data.

explicit_zero_outbreaks = (
    df_nigeria["new_outbreaks"]
    .eq(0)
)

print(
    "\nExplicit zero outbreak records: "
    f"{explicit_zero_outbreaks.sum():,}"
)


# ------------------------------------------------------------
# 8. CHECK POSITIVE OUTBREAK COUNTS
# ------------------------------------------------------------
# Positive values represent records where new outbreaks were
# explicitly reported.

positive_outbreaks = (
    df_nigeria["new_outbreaks"]
    .gt(0)
)

print(
    "Positive outbreak-count records: "
    f"{positive_outbreaks.sum():,}"
)


# ------------------------------------------------------------
# 9. SUMMARISE OBSERVATION STATUS
# ------------------------------------------------------------

print("\nOutbreak observation status:")

display(
    df_nigeria[
        "outbreak_observation_status"
    ]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 10. DISEASE DISTRIBUTION
# ------------------------------------------------------------
# Examine the diseases represented among Nigerian records.

disease_summary = (
    df_nigeria
    .groupby(
        "disease",
        dropna=False
    )
    .size()
    .reset_index(
        name="record_count"
    )
    .sort_values(
        "record_count",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nDisease distribution:")

display(disease_summary)


# ------------------------------------------------------------
# 11. TEMPORAL COVERAGE
# ------------------------------------------------------------
# Inspect the year range represented in the Nigeria subset.

print("\nTemporal coverage:")

print(
    f"Minimum year: "
    f"{df_nigeria['year'].min()}"
)

print(
    f"Maximum year: "
    f"{df_nigeria['year'].max()}"
)

print(
    f"Unique years: "
    f"{df_nigeria['year'].nunique()}"
)


# ------------------------------------------------------------
# 12. YEAR-SEMESTER DISTRIBUTION
# ------------------------------------------------------------
# This provides the temporal structure that will later be
# used for lag features and chronological validation.

year_semester_summary = (
    df_nigeria
    .groupby(
        ["year", "semester"],
        dropna=False
    )
    .size()
    .reset_index(
        name="record_count"
    )
    .sort_values(
        ["year", "semester"]
    )
)

print("\nRecords by year and semester:")

display(year_semester_summary)


# ------------------------------------------------------------
# 13. ADMINISTRATIVE GEOGRAPHY
# ------------------------------------------------------------
# Examine the administrative divisions represented in the
# Nigerian surveillance records.

administrative_summary = (
    df_nigeria
    .groupby(
        "administrative_division",
        dropna=False
    )
    .size()
    .reset_index(
        name="record_count"
    )
    .sort_values(
        "record_count",
        ascending=False
    )
)

print("\nAdministrative-division distribution:")

display(administrative_summary.head(20))


# ------------------------------------------------------------
# 14. CREATE A SIMPLE ANALYTICAL RECORD ID
# ------------------------------------------------------------
# The source contains Event_id and Outbreak_id fields, but
# these may contain unavailable values.
#
# We therefore create a unique row-level identifier for the
# analytical table. This does not replace the source IDs.

df_nigeria = (
    df_nigeria
    .reset_index(drop=True)
)

df_nigeria.insert(
    0,
    "analytical_record_id",
    range(1, len(df_nigeria) + 1)
)


# ------------------------------------------------------------
# 15. REORDER IMPORTANT ANALYTICAL VARIABLES
# ------------------------------------------------------------
# Put the main temporal, geographic, disease and observation
# fields near the beginning of the analytical table.

priority_columns = [
    "analytical_record_id",
    "year",
    "semester",
    "country",
    "world_region",
    "administrative_division",
    "disease",
    "animal_category",
    "species",
    "new_outbreaks",
    "outbreak_count_observed",
    "outbreak_observation_status",
]

remaining_columns = [
    column
    for column in df_nigeria.columns
    if column not in priority_columns
]

df_analytical = df_nigeria[
    priority_columns + remaining_columns
].copy()


# ------------------------------------------------------------
# 16. FINAL ANALYTICAL-TABLE SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("WAHIS ANALYTICAL TABLE SUMMARY")
print("=" * 60)

print(
    f"Total Nigerian records: "
    f"{len(df_analytical):,}"
)

print(
    f"Observed outbreak-count records: "
    f"{df_analytical['outbreak_count_observed'].sum():,}"
)

print(
    f"Unavailable outbreak-count records: "
    f"{(~df_analytical['outbreak_count_observed']).sum():,}"
)

print(
    f"Explicit zero outbreak records: "
    f"{df_analytical['new_outbreaks'].eq(0).sum():,}"
)

print(
    f"Positive outbreak records: "
    f"{df_analytical['new_outbreaks'].gt(0).sum():,}"
)

print("=" * 60)


# ------------------------------------------------------------
# 17. SAVE ANALYTICAL TABLE
# ------------------------------------------------------------
# Save the analytical table separately from both the raw and
# cleaned datasets.

df_analytical.to_csv(
    ANALYTICAL_DATA_PATH,
    index=False
)

print(
    "\nAnalytical table saved to:"
)

print(
    ANALYTICAL_DATA_PATH
)


# ------------------------------------------------------------
# 18. DISPLAY FINAL PREVIEW
# ------------------------------------------------------------

print("\nAnalytical-table preview:")

display(
    df_analytical.head(10)
)

Cleaned WAHIS dataset loaded.
Rows: 1,180
Columns: 19

Country distribution:


country
Nigeria    1180
Name: count, dtype: int64


Nigeria records: 1,180

New outbreaks — missing values:
735

New outbreaks — observed values:
445

New outbreaks — descriptive statistics:


count    445.000000
mean       6.840449
std       12.872353
min        1.000000
25%        1.000000
50%        3.000000
75%        7.000000
max      184.000000
Name: new_outbreaks, dtype: float64


Explicit zero outbreak records: 0
Positive outbreak-count records: 445

Outbreak observation status:


outbreak_observation_status
Unavailable    735
Observed       445
Name: count, dtype: int64

... (4 outputs left out)


Administrative-division distribution:


,administrative_division,record_count
31,Osun,70
17,Jigawa,66
18,Kaduna,65
33,Plateau,57
12,Ekiti,55
23,Kwara,52
15,Gombe,50
22,Kogi,46
27,Niger,45
6,Benue,43



WAHIS ANALYTICAL TABLE SUMMARY
Total Nigerian records: 1,180
Observed outbreak-count records: 445
Unavailable outbreak-count records: 735
Explicit zero outbreak records: 0
Positive outbreak records: 445

Analytical table saved to:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\processed\wahis_analytical_table.csv

Analytical-table preview:


,analytical_record_id,year,semester,country,world_region,administrative_division,disease,animal_category,species,new_outbreaks,...,serotype_subtype_genotype,event_id,outbreak_id,susceptible,measuring_units,cases,killed_and_disposed_of,slaughtered,deaths,vaccinated
0,1,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Fowl pox (-2005),Domestic,Birds,NaN,...,NaN,NaN,NaN,90.0,Animal,60.0,0.0,0.0,0.0,0.0
1,2,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Fowl typhoid,Domestic,Birds,NaN,...,NaN,NaN,NaN,1000.0,Animal,800.0,0.0,0.0,25.0,0.0
2,3,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Infectious bursal disease (Gumboro disease),Domestic,Birds,NaN,...,NaN,NaN,NaN,200.0,Animal,10.0,0.0,0.0,0.0,0.0
3,4,2005,Jul-Dec 2005,Nigeria,Africa,Nigeria,Newcastle disease virus (Inf. with),Domestic,Birds,NaN,...,NaN,NaN,NaN,22100.0,Animal,3527.0,0.0,800.0,2076.0,0.0
4,5,2005,Jan-Jun 2005,Nigeria,Africa,Nigeria,Newcastle disease virus (Inf. with),Domestic,Birds,NaN,...,NaN,NaN,NaN,8100.0,Animal,3430.0,0.0,0.0,912.0,13000.0
5,6,2007,Jul-Dec 2007,Nigeria,Africa,Gombe,Infectious bursal disease (Gumboro disease),Domestic,Birds,NaN,...,NaN,NaN,NaN,300.0,Animal,NaN,0.0,0.0,25.0,NaN
6,7,2008,Jan-Jun 2008,Nigeria,Africa,Adamawa,Newcastle disease virus (Inf. with),Domestic,Birds,NaN,...,NaN,NaN,NaN,213.0,Animal,NaN,NaN,NaN,32.0,NaN
7,8,2008,Jan-Jun 2008,Nigeria,Africa,Bauchi,Infectious bursal disease (Gumboro disease),Domestic,Birds,NaN,...,NaN,NaN,NaN,2040.0,Animal,969.0,0.0,55.0,545.0,NaN
8,9,2008,Jan-Jun 2008,Nigeria,Africa,Bauchi,Newcastle disease virus (Inf. with),Domestic,Birds,NaN,...,NaN,NaN,NaN,85.0,Animal,922.0,0.0,NaN,297.0,NaN
9,10,2008,Jan-Jun 2008,Nigeria,Africa,Delta,Fowl typhoid,Domestic,Birds,NaN,...,NaN,NaN,NaN,350.0,Animal,50.0,0.0,NaN,20.0,NaN


In [8]:
# ============================================================
# NOTEBOOK 03 — ANALYTICAL TABLE VALIDATION
# ============================================================

# These checks verify that the analytical table preserves
# the key methodological assumptions of the project.


# ------------------------------------------------------------
# Check 1 — The table should contain only Nigeria records.
# ------------------------------------------------------------

assert (
    df_analytical["country"]
    .str.lower()
    .eq("nigeria")
    .all()
)


# ------------------------------------------------------------
# Check 2 — Missing outbreak counts must remain missing.
# They must NOT have been converted to zero.
# ------------------------------------------------------------

missing_outbreaks = (
    df_analytical["new_outbreaks"]
    .isna()
)

assert (
    df_analytical.loc[
        missing_outbreaks,
        "outbreak_observation_status"
    ]
    .eq("Unavailable")
    .all()
)


# ------------------------------------------------------------
# Check 3 — Observed outbreak counts must be marked as
# observed.
# ------------------------------------------------------------

observed_outbreaks = (
    df_analytical["new_outbreaks"]
    .notna()
)

assert (
    df_analytical.loc[
        observed_outbreaks,
        "outbreak_observation_status"
    ]
    .eq("Observed")
    .all()
)


# ------------------------------------------------------------
# Check 4 — Explicit zero values must remain zero.
# ------------------------------------------------------------

zero_outbreaks = (
    df_analytical["new_outbreaks"]
    .eq(0)
)

assert (
    df_analytical.loc[
        zero_outbreaks,
        "new_outbreaks"
    ]
    .eq(0)
    .all()
)


# ------------------------------------------------------------
# Check 5 — The analytical record ID must be unique.
# ------------------------------------------------------------

assert (
    df_analytical["analytical_record_id"]
    .is_unique
)


# ------------------------------------------------------------
# Check 6 — No duplicate complete rows.
# ------------------------------------------------------------

duplicate_rows = (
    df_analytical
    .duplicated()
    .sum()
)

print(
    f"Duplicate complete analytical rows: "
    f"{duplicate_rows:,}"
)


# ------------------------------------------------------------
# Final validation result
# ------------------------------------------------------------

print("\nAnalytical-table validation completed successfully.")

Duplicate complete analytical rows: 0

Analytical-table validation completed successfully.


# Analytical Table Conclusion

The cleaned WAHIS data has been transformed into a Nigeria-specific analytical table.

The table explicitly distinguishes between:

- observed outbreak-count information;
- explicitly reported zero outbreak counts; and
- unavailable outbreak-count information.

Unavailable outbreak information remains missing and is not converted into a negative disease label.

This distinction is necessary because the objective is to develop an early-warning system rather than to assume that every unreported surveillance observation represents disease absence.

The analytical table is now ready for the next stage:

**04 — Exploratory Analysis and Surveillance Patterns**

The next stage will investigate disease frequency, temporal patterns, geographic distribution and outbreak-count behaviour before feature engineering and machine-learning model development.